# 00 · Démo : corrigé et réponses

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/00_setup/demo_solutions.ipynb)

Notebook de **solutions** de la démo. Chaque cellule taguée `answer` appelle `wb.record(...)`, qui enregistre une empreinte de la bonne réponse ; `python tools/build_answers.py` récolte ces empreintes dans `src/wb/answers.json`. C'est le même mécanisme que pour tous les chapitres.

In [1]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast

🧪 Deep Learning Workbook : environnement
   Plateforme : Local (Linux x86_64, 2 CPU)
   Python 3.13.13 | numpy 2.1.3 | pandas 2.2.3 | matplotlib 3.10.0 | scikit-learn 1.6.1 | scipy 1.16.3 | torch 2.11.0+cpu | torchvision 0.26.0+cpu
   Device : cpu | Graine : 42 | FAST_MODE : True


In [2]:
penguins = wb.datasets.load_penguins()
flippers = penguins["flipper_length_mm"]

**Ex demo.1** : nombre de manchots.

In [3]:
wb.record("demo.1", len(penguins))

WB_ANSWER {"hash": "49e24a9d7125d57a1d8bdcf680d37aeb831e37274bb477213eec9e8f88659ca5", "id": "demo.1", "kind": "int", "magnitude": 2, "sign": 1}
📝 Ex demo.1 : réponse enregistrée (int).


{'kind': 'int',
 'hash': '49e24a9d7125d57a1d8bdcf680d37aeb831e37274bb477213eec9e8f88659ca5',
 'sign': 1,
 'magnitude': 2}

**Ex demo.2** : longueur moyenne des nageoires, 1 décimale, valeurs manquantes ignorées. `Series.mean()` ignore les `NaN` par défaut. Erreurs classiques : la somme au lieu de la moyenne ; remplacer les valeurs manquantes par 0.

In [4]:
wb.record(
    "demo.2",
    flippers.mean(),
    decimals=1,
    mistakes={
        "tu as calculé une somme ; la moyenne divise par le nombre de valeurs": flippers.sum(),
        "les valeurs manquantes ont été comptées comme des zéros ; ignore-les (dropna)": flippers.fillna(0).mean(),
    },
)

WB_ANSWER {"decimals": 1, "hash": "bb211dcac15efc21391dd7756467bd1cb2edb16d208e5989b86ac70ff8017c0c", "hash_coarse": "42b3ff493da953b2f1259f6934b2bf4e9597c9fbb514365dc4619676890ae56b", "id": "demo.2", "kind": "float", "magnitude": 2, "mistakes": {"5e78082cb49af22346bd23d7f2d7443a2acda08005707fa1ee953da6f3804f9b": "tu as calculé une somme ; la moyenne divise par le nombre de valeurs", "5f8df0888c6a269c4d869158ed892b16e39f7ad0029e3dbe9ade0616a47d062a": "les valeurs manquantes ont été comptées comme des zéros ; ignore-les (dropna)"}, "sign": 1}
📝 Ex demo.2 : réponse enregistrée (float, 1 décimale(s)).


{'kind': 'float',
 'hash': 'bb211dcac15efc21391dd7756467bd1cb2edb16d208e5989b86ac70ff8017c0c',
 'decimals': 1,
 'sign': 1,
 'magnitude': 2,
 'hash_coarse': '42b3ff493da953b2f1259f6934b2bf4e9597c9fbb514365dc4619676890ae56b',
 'mistakes': {'5e78082cb49af22346bd23d7f2d7443a2acda08005707fa1ee953da6f3804f9b': 'tu as calculé une somme ; la moyenne divise par le nombre de valeurs',
  '5f8df0888c6a269c4d869158ed892b16e39f7ad0029e3dbe9ade0616a47d062a': 'les valeurs manquantes ont été comptées comme des zéros ; ignore-les (dropna)'}}

**Ex demo.3** : espèce la plus lourde en moyenne.

In [5]:
mean_mass = penguins.groupby("species")["body_mass_g"].mean()
print(mean_mass.round(0))
wb.record("demo.3", mean_mass.idxmax())

species
Adelie       3701.0
Chinstrap    3733.0
Gentoo       5076.0
Name: body_mass_g, dtype: float64
WB_ANSWER {"hash": "8b0a725c0af5540ad381fd5d4a5041d37e358e36b3e2deed8dadee476fc19c5e", "id": "demo.3", "kind": "str"}
📝 Ex demo.3 : réponse enregistrée (str).


{'kind': 'str',
 'hash': '8b0a725c0af5540ad381fd5d4a5041d37e358e36b3e2deed8dadee476fc19c5e'}

**Ex demo.4** : nombre d'îles distinctes.

In [6]:
def count_islands(df):
    """Return the number of distinct islands in the dataset."""
    return df["island"].nunique()


wb.record("demo.4", count_islands(penguins))

WB_ANSWER {"hash": "afad62b675558ea456c78dbd768b7320957cb99c87cc68aa986e98112d9018ad", "id": "demo.4", "kind": "int", "magnitude": 0, "sign": 1}
📝 Ex demo.4 : réponse enregistrée (int).


{'kind': 'int',
 'hash': 'afad62b675558ea456c78dbd768b7320957cb99c87cc68aa986e98112d9018ad',
 'sign': 1,
 'magnitude': 0}

**mylearn** : la référence passe ses tests.

In [7]:
result = subprocess.run([sys.executable, "-m", "pytest", "tests/test_example_mylearn.py", "--impl=ref",
                         "-q", "-p", "no:cacheprovider", "--color=no"], cwd=ROOT, capture_output=True, text=True)
print(result.stdout.strip().splitlines()[-1])

5 passed in 0.02s
